In [18]:
# ---------------------- IMPORTS ----------------------
import re
import os
import math
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ---------------------- CONFIG ----------------------
TOP_K_BRAND = 100
TOP_K_FLAVOR = 100

# ---------------------- HELPERS ----------------------
weight_re = re.compile(
    r'(\d+(?:\.\d+)?)(?:\s*-\s*\d+(?:\.\d+)?)?\s*(fl oz|fluid ounce|fluid ounces|oz|ounce|ounces|g\b|gram|grams|kg|kilogram|kilograms|ml|milliliter|milliliters|l\b|liter|liters|lb\b|pound|pounds)\b',
    re.IGNORECASE
)

def parse_weight_to_grams(text):
    if pd.isna(text):
        return (None, None)
    m = weight_re.search(str(text))
    if not m:
        return (None, None)
    val = float(m.group(1))
    unit = m.group(2).lower()
    unit_norm = unit
    if 'fl' in unit:
        unit_norm = 'fl oz'
    elif 'ounce' in unit or unit.strip() == 'oz':
        unit_norm = 'oz'
    elif unit.startswith('g'):
        unit_norm = 'g'
    elif 'kg' in unit:
        unit_norm = 'kg'
    elif 'ml' in unit:
        unit_norm = 'ml'
    elif unit.startswith('l'):
        unit_norm = 'l'
    elif unit.startswith('lb') or 'pound' in unit:
        unit_norm = 'lb'
    grams = None
    if unit_norm == 'oz':
        grams = val * 28.349523125
    elif unit_norm == 'g':
        grams = val
    elif unit_norm == 'kg':
        grams = val * 1000.0
    elif unit_norm == 'ml':
        grams = val * 1.0
    elif unit_norm == 'l':
        grams = val * 1000.0
    elif unit_norm == 'lb':
        grams = val * 453.59237
    elif unit_norm == 'fl oz':
        grams = val * 29.5735
    if grams is not None:
        return (f"{val} {unit_norm}", round(grams, 2))
    return (f"{val} {unit_norm}", None)

def extract_item_name(text):
    if pd.isna(text):
        return None
    m = re.search(r'Item Name[:\-]?\s*(.?)(?=(\s(Brand|Flavor|UPC|Size|Weight|Manufacturer|$)))', str(text), re.IGNORECASE)
    if m:
        return m.group(1).strip(" ,.-\n\t")
    first = str(text).split(',')[0]
    return first.strip()

def extract_brand(text):
    if pd.isna(text):
        return None
    m = re.search(r'Brand[:\-]?\s*([A-Za-z0-9&\'’\-.\s]{1,80})(?=(\s*(Flavor|UPC|Item|Weight|$)))', str(text), re.IGNORECASE)
    if m:
        return m.group(1).strip(" ,.-\n\t")
    m2 = re.search(r'Manufacturer[:\-]?\s*([A-Za-z0-9&\'’\-.\s]{1,80})', str(text), re.IGNORECASE)
    if m2:
        return m2.group(1).strip(" ,.-\n\t")
    return None

def extract_flavor(text):
    if pd.isna(text):
        return None
    m = re.search(r'Flavor[:\-]?\s*([A-Za-z0-9&\'’\-\s]{1,40})', str(text), re.IGNORECASE)
    if m:
        return m.group(1).strip(" ,.-\n\t")
    kws = ['vanilla','chocolate','strawberry','mild','spicy','hot','sweet','sour','apple','cider','garlic','lemon','honey','barbecue','bbq','salted','unsalted','original','plain','roasted','smoked','caramel','mint','vinegar']
    txt = str(text).lower()
    for kw in kws:
        if re.search(r'\b' + re.escape(kw) + r'\b', txt):
            return kw
    return None

def top_k_or_other(series, k=100):
    top = series.fillna('Unknown').value_counts().nlargest(k).index
    return series.fillna('Unknown').apply(lambda x: x if x in top else 'Other')

def smape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    denom = np.where(denom == 0, 1e-8, denom)
    return np.mean(np.abs(y_pred - y_true) / denom) * 100.0

# ---------------------- LOAD DATA ----------------------
TRAIN_FILENAME = 'train.csv'
TEST_FILENAME = 'test.csv'

if not os.path.exists(TRAIN_FILENAME):
    raise FileNotFoundError(f"train file not found at {TRAIN_FILENAME}. Place train.csv in the current directory and retry.")
if not os.path.exists(TEST_FILENAME):
    raise FileNotFoundError(f"test file not found at {TEST_FILENAME}. Place test.csv in the current directory and retry.")

print('Loading train and test...')
train = pd.read_csv(TRAIN_FILENAME)
test = pd.read_csv(TEST_FILENAME)

print('Train shape:', train.shape)
print('Test shape:', test.shape)

# ---------------------- TEXT COLUMN DETECTION ----------------------
text_col = None
for c in ['catalog_content','catalog content','description','item_description','catalog','product_description']:
    if c in train.columns:
        text_col = c
        break
if text_col is None:
    for c in train.columns:
        if train[c].dtype == object and train[c].dropna().str.len().median() > 20:
            text_col = c
            break
if text_col is None:
    raise RuntimeError('Could not detect the textual product column in train.csv. Rename to catalog_content or description')

print('Using text column:', text_col)

# ---------------------- FEATURE ENGINEERING ----------------------
train['item_name'] = train[text_col].apply(extract_item_name)
train['brand'] = train[text_col].apply(extract_brand)
train['flavor'] = train[text_col].apply(extract_flavor)
wt = train[text_col].apply(parse_weight_to_grams)
train['weight_text_raw'] = wt.apply(lambda x: x[0])
train['weight_grams_text'] = wt.apply(lambda x: x[1])
train['is_organic'] = train[text_col].str.contains('organic', case=False, na=False).astype(int)
train['is_gluten_free'] = train[text_col].str.contains('gluten\s*free', case=False, na=False).astype(int)
train['weight_image_raw'] = None
train['weight_grams_image'] = np.nan

test_text_col = text_col if text_col in test.columns else None
if not test_text_col:
    for c in ['catalog_content','catalog content','description','item_description','catalog','product_description']:
        if c in test.columns:
            test_text_col = c
            break
    else:
        for c in test.columns:
            if test[c].dtype == object:
                test_text_col = c
                break
if test_text_col is None:
    raise RuntimeError('Could not detect the textual product column in test.csv')

print('Using text column in test:', test_text_col)

test['item_name'] = test[test_text_col].apply(extract_item_name)
test['brand'] = test[test_text_col].apply(extract_brand)
test['flavor'] = test[test_text_col].apply(extract_flavor)
wt_t = test[test_text_col].apply(parse_weight_to_grams)
test['weight_text_raw'] = wt_t.apply(lambda x: x[0])
test['weight_grams_text'] = wt_t.apply(lambda x: x[1])
test['is_organic'] = test[test_text_col].str.contains('organic', case=False, na=False).astype(int)
test['is_gluten_free'] = test[test_text_col].str.contains('gluten\s*free', case=False, na=False).astype(int)
test['weight_image_raw'] = None
test['weight_grams_image'] = np.nan

# ---------------------- MERGE WEIGHT COLUMNS ----------------------
train['weight_grams_combined'] = train['weight_grams_text'].fillna(train['weight_grams_image'])
test['weight_grams_combined'] = test['weight_grams_text'].fillna(test['weight_grams_image'])

train['weight_final'] = train['weight_grams_combined']
test['weight_final'] = test['weight_grams_combined']

# ---------------------- HANDLE MISSING WEIGHTS ----------------------
train_missing_pct = train['weight_final'].isna().mean()
THRESHOLD = 0.2
median_train = train['weight_final'].median()
test['weight_final'] = test['weight_grams_combined'].fillna(median_train)

# ---------------------- TOP K CATEGORICAL ----------------------
train['brand_top'] = top_k_or_other(train['brand'], k=TOP_K_BRAND)
train['flavor_top'] = top_k_or_other(train['flavor'], k=TOP_K_FLAVOR)
train['item_text'] = train['item_name'].fillna('')

FEATURE_ITEM = 'item_text'
FEATURE_BRAND = 'brand_top'
FEATURE_FLAVOR = 'flavor_top'
NUM_FEATURES = ['weight_final','is_organic','is_gluten_free']

X = train[[FEATURE_ITEM, FEATURE_BRAND, FEATURE_FLAVOR] + NUM_FEATURES].copy()
y = train['price'].astype(float)

# ---------------------- ENCODING AND PIPELINE ----------------------
preprocessor = ColumnTransformer(transformers=[
    ('tfidf', TfidfVectorizer(max_features=500, ngram_range=(1,2), stop_words='english'), FEATURE_ITEM),
    ('brand_ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), [FEATURE_BRAND]),
    ('flav_ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), [FEATURE_FLAVOR]),
    ('num', StandardScaler(), NUM_FEATURES)
], remainder='drop', sparse_threshold=0)

pipeline = Pipeline([
    ('preproc', preprocessor),
    ('reg', HistGradientBoostingRegressor(random_state=42))
])

# ---------------------- TRAIN ----------------------
print('\nTraining model on full train data...')
pipeline.fit(X, y)
y_pred_train = pipeline.predict(X)

r2_train = r2_score(y, y_pred_train)
rmse_train = math.sqrt(mean_squared_error(y, y_pred_train))
mae_train = mean_absolute_error(y, y_pred_train)
smape_train = smape(y.values, y_pred_train)

print(f"Train R2: {r2_train:.4f}")
print(f"Train RMSE: {rmse_train:.4f}")
print(f"Train MAE: {mae_train:.4f}")
print(f"Train SMAPE: {smape_train:.4f}")

# ---------------------- PREPARE TEST FOR PREDICTION ----------------------
if 'sample_id' not in test.columns:
    test['sample_id'] = np.arange(len(test))

train_brands = set(train['brand_top'].unique())
train_flavors = set(train['flavor_top'].unique())

test['brand_top'] = test['brand'].fillna('Unknown').apply(lambda x: x if x in train_brands else 'Other')
test['flavor_top'] = test['flavor'].fillna('Unknown').apply(lambda x: x if x in train_flavors else 'Other')
test['item_name'] = test['item_name'].fillna('').astype(str)
test['item_text'] = test['item_name']

for col in NUM_FEATURES:
    if col not in test.columns:
        test[col] = 0
    test[col] = pd.to_numeric(test[col], errors='coerce').fillna(0)

X_test = test[[FEATURE_ITEM, FEATURE_BRAND, FEATURE_FLAVOR] + NUM_FEATURES].copy()

# ---------------------- PREDICTION ----------------------
print('Predicting on test data...')
preds = pipeline.predict(X_test)
preds = np.maximum(preds, 0.01)
test['price'] = np.round(preds, 2)

# ---------------------- SAVE OUTPUT ----------------------
test[['sample_id', 'price']].to_csv('test_out.csv', index=False)
train.to_csv('train_processed_for_model.csv', index=False)

print('Done. Outputs saved:')
print('- test_out.csv with sample_id and predicted price')
print('- train_processed_for_model.csv with full train data')

Loading train and test...
Train shape: (75000, 4)
Test shape: (75000, 3)
Using text column: catalog_content
Using text column in test: catalog_content

Training model on full train data...
Train R2: 0.2921
Train RMSE: 28.0822
Train MAE: 14.8581
Train SMAPE: 67.6877
Predicting on test data...
Done. Outputs saved:
- test_out.csv with sample_id and predicted price
- train_processed_for_model.csv with full train data
